In [10]:
import re
import math
from collections import deque
from urllib.parse import urljoin, urlparse

import httpx
from bs4 import BeautifulSoup
import trafilatura
import numpy as np
from sentence_transformers import SentenceTransformer

# ===== Config =====

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

MAX_PAGES_PER_DOMAIN = 50          # max pages to crawl per seed site
REQUEST_TIMEOUT = 10.0             # seconds
MIN_ARTICLE_CHARS = 200            # minimal length for extracted article text
MAX_CHUNKS_PER_ARTICLE = 32        # cap chunks per article
TOP_K = 10                         # how many top matches to show

DEBUG = True  # set False when stable

model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)


Model loaded: sentence-transformers/all-MiniLM-L6-v2


In [11]:
def fetch_html(url: str) -> str | None:
    """
    Fetch URL and return HTML content if it's an HTML page.
    """
    try:
        r = httpx.get(url, timeout=REQUEST_TIMEOUT, follow_redirects=True)
        if r.status_code == 200 and "text/html" in r.headers.get("content-type", ""):
            return r.text
        if DEBUG:
            print(f"[fetch_html] Non-HTML or bad status for {url}: {r.status_code}")
    except Exception as e:
        if DEBUG:
            print(f"[fetch_html] Error fetching {url}: {e}")
    return None


def is_junk_href(href: str) -> bool:
    """
    Filter anchors, mailto, javascript, empty links.
    """
    href = href.strip()
    if not href:
        return True
    if href.startswith("#"):
        return True
    if href.lower().startswith("mailto:"):
        return True
    if href.lower().startswith("javascript:"):
        return True
    return False


def same_domain(url: str, domain: str) -> bool:
    """
    Check if url is on the given domain.
    """
    return urlparse(url).netloc == domain


In [12]:
def looks_like_article(url: str) -> bool:
    """
    Heuristic to guess if a URL is an article page.
    Tuned for common news patterns + Mako/Ynet-like URLs.
    """
    u = url.lower()

    # bad = [
    #     "/tag/", "/tags/", "/topic/", "/topics/",
    #     "/about", "/privacy", "/terms",
    #     "/account", "/signup", "/login",
    #     "/video", "/live", "/podcast",
    #     "/weather", "/tv/", "/shows/",
    # ]
    # if any(b in u for b in bad):
    #     return False

    # Strong hints
    if "article-" in u:        # e.g. mako / ynet style
        return True
    if "/article/" in u:
        return True

    path = urlparse(u).path

    # Generic heuristic: some depth + digits often means article id / date
    if path.count("/") >= 2 and any(c.isdigit() for c in path):
        return True

    return False


def looks_like_listing_page(url: str, seed_url: str) -> bool:
    """
    Guess if URL is a section/listing page worth shallow crawling.
    """
    # if looks_like_article(url):
    #     return False

    seed_depth = urlparse(seed_url).path.count("/")
    depth = urlparse(url).path.count("/")
    depth_diff = depth - seed_depth

    return depth_diff <= 3


In [13]:
def extract_main_text(url: str, html: str | None = None) -> str | None:
    """
    Extract main article text using trafilatura.
    Returns None if extraction fails or content is too short.
    """
    try:
        if html is None:
            downloaded = trafilatura.fetch_url(url)
            if not downloaded:
                if DEBUG:
                    print(f"[extract_main_text] fetch_url failed for {url}")
                return None
            extracted = trafilatura.extract(
                downloaded,
                include_comments=False,
                include_tables=False,
                favor_recall=True,
            )
        else:
            extracted = trafilatura.extract(
                html,
                include_comments=False,
                include_tables=False,
                favor_recall=True,
            )

        if not extracted:
            if DEBUG:
                print(f"[extract_main_text] no text extracted for {url}")
            return None

        text = extracted.strip()
        if len(text) < MIN_ARTICLE_CHARS:
            if DEBUG:
                print(f"[extract_main_text] too short ({len(text)} chars) for {url}")
            return None

        return text

    except Exception as e:
        if DEBUG:
            print(f"[extract_main_text] Error for {url}: {e}")
        return None


In [14]:
def clean_and_segment(text: str, max_chunks: int = MAX_CHUNKS_PER_ARTICLE) -> list[str]:
    """
    Normalize whitespace, split to sentence-like pieces,
    group into ~400+ char chunks.
    """
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []

    sentences = re.split(r"(?<=[.!?]) +", text)

    chunks = []
    buf = ""
    for s in sentences:
        if not s:
            continue

        # Grow buffer up to ~400 chars
        if len(buf) + len(s) + 1 < 400:
            buf += (" " if buf else "") + s
        else:
            if buf:
                chunks.append(buf.strip())
            buf = s

        if len(chunks) >= max_chunks:
            break

    if buf and len(chunks) < max_chunks:
        chunks.append(buf.strip())

    return chunks


def embed_article(text: str) -> np.ndarray | None:
    """
    Build a single normalized vector for an article:
    - encode chunks without per-chunk normalization
    - average
    - L2 normalize once
    """
    chunks = clean_and_segment(text)
    if not chunks:
        return None

    vecs = model.encode(chunks, convert_to_numpy=True, normalize_embeddings=False)
    v = vecs.mean(axis=0)

    norm = np.linalg.norm(v)
    if norm == 0.0 or math.isnan(norm):
        if DEBUG:
            print("[embed_article] zero/NaN norm")
        return None

    return v / norm


In [15]:
def crawl_for_candidate_articles(seed_url: str,
                                 max_pages: int = MAX_PAGES_PER_DOMAIN) -> list[str]:
    """
    Tiny BFS crawler:
    - start at seed_url
    - keep same domain
    - collect URLs that look like articles
    - follow some listing pages shallowly
    """
    domain = urlparse(seed_url).netloc

    queue = deque([seed_url])
    seen = set([seed_url])
    articles: list[str] = []

    while queue and len(seen) < max_pages:
        url = queue.popleft()
        html = fetch_html(url)
        if not html:
            continue

        soup = BeautifulSoup(html, "html.parser")

        for a in soup.find_all("a", href=True):
            href = a["href"].strip()
            if is_junk_href(href):
                continue

            abs_url = urljoin(url, href)

            if abs_url in seen:
                continue
            if not same_domain(abs_url, domain):
                continue

            seen.add(abs_url)

            if looks_like_article(abs_url):
                articles.append(abs_url)
            elif looks_like_listing_page(abs_url, seed_url):
                queue.append(abs_url)

            if len(seen) >= max_pages:
                break

    if DEBUG:
        print(f"[crawl] {seed_url} -> {len(articles)} article-like URLs, {len(seen)} pages seen")

    return articles


In [16]:
def debug_candidates(urls: list[str]):
    """
    For debugging: show which candidates successfully extract content.
    """
    print("[DEBUG] Inspecting candidate URLs")
    for u in urls:
        text = extract_main_text(u)
        length = len(text) if text else 0
        status = "OK" if text else "FAIL"
        print(f"- {status:4} | len={length:5} | {u}")


In [17]:
def find_best_matches_for_article(source_article_url: str,
                                  seed_sites: list[str],
                                  top_k: int = TOP_K):
    # 1. Source article -> text -> vector
    print(f"[INFO] Source article: {source_article_url}")
    src_html = fetch_html(source_article_url)
    if not src_html:
        print("[ERROR] Could not fetch source article.")
        return

    src_text = extract_main_text(source_article_url, src_html)
    if not src_text:
        print("[ERROR] Could not extract main text from source article.")
        return

    query_vec = embed_article(src_text)
    if query_vec is None:
        print("[ERROR] Could not embed source article.")
        return

    # 2. Collect candidates from all seed sites
    all_candidate_urls: list[str] = []
    for seed in seed_sites:
        print(f"[INFO] Crawling: {seed}")
        urls = crawl_for_candidate_articles(seed)
        print(f"       Found {len(urls)} candidate URLs.")
        all_candidate_urls.extend(urls)

    # Deduplicate while preserving order
    all_candidate_urls = list(dict.fromkeys(all_candidate_urls))
    print(f"[INFO] Unique candidate URLs: {len(all_candidate_urls)}")

    if not all_candidate_urls:
        print("[WARN] No candidate URLs found.")
        return

    if DEBUG:
        debug_candidates(all_candidate_urls)

    # 3. For each candidate: extract + embed
    candidate_vecs: list[np.ndarray] = []
    valid_urls: list[str] = []

    for url in all_candidate_urls:
        text = extract_main_text(url)
        if not text:
            continue
        v = embed_article(text)
        if v is None:
            continue
        candidate_vecs.append(v)
        valid_urls.append(url)

    print(f"[INFO] Candidates with embeddings: {len(valid_urls)}")

    if not candidate_vecs:
        print("[WARN] No valid candidate embeddings.")
        return

    # 4. Similarity + ranking
    mat = np.stack(candidate_vecs, axis=0)
    sims = mat @ query_vec
    order = np.argsort(-sims)

    print("\n[RESULTS] Top matches:\n")
    for rank in order[:min(top_k, len(valid_urls))]:
        score = float(sims[rank])
        url = valid_urls[rank]
        print(f"{score:.3f}  {url}")


In [19]:
SOURCE_ARTICLE_URL = "https://www.ynet.co.il/news/article/s1xgij6yzg#autoplay"
SEED_SITES = [
   "https://www.mako.co.il/",
   "https://www.n12.co.il/"
]

find_best_matches_for_article(SOURCE_ARTICLE_URL, SEED_SITES, top_k=TOP_K)


[INFO] Source article: https://www.ynet.co.il/news/article/s1xgij6yzg#autoplay
[INFO] Crawling: https://www.mako.co.il/
[crawl] https://www.mako.co.il/ -> 6 article-like URLs, 50 pages seen
       Found 6 candidate URLs.
[INFO] Crawling: https://www.n12.co.il/
[crawl] https://www.n12.co.il/ -> 1 article-like URLs, 26 pages seen
       Found 1 candidate URLs.
[INFO] Unique candidate URLs: 7
[DEBUG] Inspecting candidate URLs
- OK   | len=  315 | https://www.mako.co.il/culture-tv/articles/Article-c75a4149b6ef091027.htm
- OK   | len=  315 | https://www.mako.co.il/spirituality-popular_culture/Article-494842ff06e9431006.htm
- OK   | len=  315 | https://www.mako.co.il/home-family-kids/Article-df5ebe37893f281026.htm
- OK   | len=  315 | https://www.mako.co.il/spirituality-popular_culture/holidays/Article-52fa2d25f1df391026.htm
- OK   | len=  315 | https://www.mako.co.il/travel-israel/magazine/Article-8cfb2c8898caf71027.htm
- OK   | len=  315 | https://www.mako.co.il/help-sitemap/Article-cd0906